# Bayesian Poisson Model for World Cup Goals

This notebook supports the presentation. It keeps the project focused on a Mathematical and Statistical Foundations goal: explaining Bayesian inference for count data.

The football application is total goals per World Cup match. We use recent completed tournaments to estimate the average goal rate, then check whether the 2022 World Cup goal counts are plausible under the model.

## Model

Let `Y` be the total number of goals in one match.

$$Y_i \mid \lambda \sim Poisson(\lambda)$$

The unknown parameter `lambda` is the expected number of goals per match.

For Bayesian inference, we place a Gamma prior on `lambda`:

$$\lambda \sim Gamma(\alpha, \beta)$$

Using the rate parameterization, the prior mean is:

$$E[\lambda] = \frac{\alpha}{\beta}$$

## Posterior Update

The Gamma prior is conjugate to the Poisson likelihood. If we observe `n` matches and total goals `sum_y`, then:

$$\lambda \mid data \sim Gamma(\alpha + \sum y_i, \beta + n)$$

This is the key mathematical result used in the notebook.

In [ ]:
from pathlib import Path
import json
import math
import statistics

ROOT = Path.cwd()
DATA_DIR = ROOT / "worldcup.json"

def load_worldcup_matches(year):
    path = DATA_DIR / str(year) / "worldcup.json"
    with path.open(encoding="utf-8") as file:
        data = json.load(file)
    return data["matches"]

def scored_full_time_matches(year):
    matches = load_worldcup_matches(year)
    return [match for match in matches if "score" in match and "ft" in match["score"]]

def total_goals_by_match(year):
    return [sum(match["score"]["ft"]) for match in scored_full_time_matches(year)]

def summarize_goals(years):
    rows = []
    for year in years:
        goals = total_goals_by_match(year)
        rows.append({
            "year": year,
            "matches": len(goals),
            "total_goals": sum(goals),
            "avg_goals": sum(goals) / len(goals),
        })
    return rows

def frequency_table(values):
    counts = {}
    for value in values:
        counts[value] = counts.get(value, 0) + 1
    return dict(sorted(counts.items()))

def ascii_bar_chart(rows, label_key, value_key, width=32):
    max_value = max(row[value_key] for row in rows) if rows else 0
    for row in rows:
        value = row[value_key]
        bar_length = round((value / max_value) * width) if max_value else 0
        bar = "#" * bar_length
        print(f"{row[label_key]!s:>6} | {bar:<{width}} {value:.3f}")

def credible_interval(probability_rows, mass=0.90):
    lower_tail = (1 - mass) / 2
    upper_tail = 1 - lower_tail
    cumulative = 0.0
    lower = None
    upper = None
    for row in probability_rows:
        cumulative += row["probability"]
        if lower is None and cumulative >= lower_tail:
            lower = row["goals"]
        if upper is None and cumulative >= upper_tail:
            upper = row["goals"]
            break
    return lower, upper

## Load and Inspect the Data

The main model uses 2010, 2014, and 2018 as training data. The 2022 World Cup is held out as the validation example.

In [ ]:
train_years = [2010, 2014, 2018]
test_year = 2022

summary_rows = summarize_goals(train_years + [test_year])
for row in summary_rows:
    print(
        f"{row['year']}: {row['matches']} matches, "
        f"{row['total_goals']} goals, "
        f"average = {row['avg_goals']:.3f}"
    )

print("\nAverage goals per tournament:")
ascii_bar_chart(summary_rows, "year", "avg_goals")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 4))
    plt.bar([str(row["year"]) for row in summary_rows], [row["avg_goals"] for row in summary_rows])
    plt.axhline(2.5, color="tab:red", linestyle="--", label="Prior mean = 2.5")
    plt.title("Average Goals per Match by World Cup")
    plt.xlabel("Tournament")
    plt.ylabel("Average goals per match")
    plt.legend()
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib is unavailable; text chart shown instead.")

## Prior Choice

We use a weakly informative prior centered around 2.5 goals per match:

$$\lambda \sim Gamma(5, 2)$$

The prior mean is `5 / 2 = 2.5`. This is reasonable for modern World Cup matches, but the prior is not very strong compared with 192 training matches.

In [ ]:
alpha_prior = 5.0
beta_prior = 2.0

train_goals = []
for year in train_years:
    train_goals.extend(total_goals_by_match(year))

train_total_goals = sum(train_goals)
train_match_count = len(train_goals)

alpha_post = alpha_prior + train_total_goals
beta_post = beta_prior + train_match_count

prior_mean = alpha_prior / beta_prior
posterior_mean = alpha_post / beta_post

print(f"Training matches: {train_match_count}")
print(f"Training total goals: {train_total_goals}")
print(f"Prior mean lambda: {prior_mean:.3f}")
print(f"Posterior alpha: {alpha_post:.3f}")
print(f"Posterior beta: {beta_post:.3f}")
print(f"Posterior mean lambda: {posterior_mean:.3f}")

print("\nPrior explanation:")
print("Gamma(5, 2) has mean 2.5, a reasonable weak starting belief for modern World Cup goals.")
print("After 192 training matches, the posterior is mostly driven by observed data, not the prior.")

try:
    import matplotlib.pyplot as plt
    import numpy as np

    def gamma_pdf(x, alpha, beta):
        log_density = alpha * math.log(beta) - math.lgamma(alpha) + (alpha - 1) * math.log(x) - beta * x
        return math.exp(log_density)

    xs = np.linspace(0.01, 5, 300)
    prior_density = [gamma_pdf(float(x), alpha_prior, beta_prior) for x in xs]
    posterior_density = [gamma_pdf(float(x), alpha_post, beta_post) for x in xs]

    plt.figure(figsize=(7, 4))
    plt.plot(xs, prior_density, label="Prior Gamma(5, 2)")
    plt.plot(xs, posterior_density, label="Posterior after 2010-2018")
    plt.title("Prior vs Posterior Belief about Lambda")
    plt.xlabel("lambda: expected goals per match")
    plt.ylabel("density")
    plt.legend()
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib/numpy unavailable; prior and posterior means printed above.")

## Posterior predictive distribution

The posterior predictive distribution answers:

> How many goals might happen in a future match, after learning from the training data?

For a Gamma-Poisson model, the predictive distribution is a negative binomial distribution. The function below computes the predictive probability for a future match having exactly `k` goals.

In [ ]:
def negative_binomial_predictive_pmf(k, alpha, beta):
    """Posterior predictive P(Y_new = k) for Poisson-Gamma model.

    Uses Gamma(shape=alpha, rate=beta). For one future match,
    P(Y=k) = C(k+alpha-1, k) * (beta/(beta+1))^alpha * (1/(beta+1))^k.
    The log-gamma form supports non-integer alpha.
    """
    log_coef = math.lgamma(k + alpha) - math.lgamma(alpha) - math.lgamma(k + 1)
    log_prob = log_coef + alpha * math.log(beta / (beta + 1)) + k * math.log(1 / (beta + 1))
    return math.exp(log_prob)

predictive_probs = {
    goals: negative_binomial_predictive_pmf(goals, alpha_post, beta_post)
    for goals in range(0, 9)
}

for goals, probability in predictive_probs.items():
    print(f"P(next match has {goals} goals) = {probability:.4f}")

print(f"P(next match has 9+ goals) = {1 - sum(predictive_probs.values()):.4f}")

predictive_probability_rows = [
    {"goals": goals, "probability": probability}
    for goals, probability in predictive_probs.items()
]
interval_90 = credible_interval(predictive_probability_rows, mass=0.90)
print(f"90% posterior predictive credible interval for one match: {interval_90[0]} to {interval_90[1]} goals")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 4))
    plt.bar(list(predictive_probs.keys()), list(predictive_probs.values()))
    plt.title("Posterior Predictive Distribution")
    plt.xlabel("Total goals in a future match")
    plt.ylabel("Predicted probability")
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib unavailable; predictive probabilities printed above.")

## Validate on 2022

The simple model gives the same posterior predictive distribution for every future match because it does not know team strength, opponent strength, or tournament stage. That is a limitation, but it keeps the Bayesian update easy to explain.

In [ ]:
test_goals = total_goals_by_match(test_year)
test_average = sum(test_goals) / len(test_goals)

absolute_errors = [abs(goal_count - posterior_mean) for goal_count in test_goals]
squared_errors = [(goal_count - posterior_mean) ** 2 for goal_count in test_goals]

mae = sum(absolute_errors) / len(absolute_errors)
rmse = math.sqrt(sum(squared_errors) / len(squared_errors))

def poisson_pmf(k, lam):
    return math.exp(-lam + k * math.log(lam) - math.lgamma(k + 1))

def predictive_probability_for_observed(k):
    return negative_binomial_predictive_pmf(k, alpha_post, beta_post)

mean_log_predictive_probability = statistics.mean(
    math.log(predictive_probability_for_observed(goal_count))
    for goal_count in test_goals
)

actual_frequency = frequency_table(test_goals)
actual_vs_predicted_rows = []
for goals in range(0, max(max(actual_frequency), max(predictive_probs)) + 1):
    actual_count = actual_frequency.get(goals, 0)
    predicted_probability = negative_binomial_predictive_pmf(goals, alpha_post, beta_post)
    actual_vs_predicted_rows.append({
        "goals": goals,
        "actual_count": actual_count,
        "actual_share": actual_count / len(test_goals),
        "predicted_probability": predicted_probability,
    })

lower_90, upper_90 = credible_interval(
    [{"goals": row["goals"], "probability": row["predicted_probability"]} for row in actual_vs_predicted_rows],
    mass=0.90,
)
coverage_90 = sum(lower_90 <= goal_count <= upper_90 for goal_count in test_goals) / len(test_goals)

print(f"2022 matches: {len(test_goals)}")
print(f"2022 average goals: {test_average:.3f}")
print(f"Posterior predictive mean: {posterior_mean:.3f}")
print(f"Mean absolute error: {mae:.3f}")
print(f"Root mean squared error: {rmse:.3f}")
print(f"Mean log predictive probability: {mean_log_predictive_probability:.3f}")
print(f"90% predictive interval: {lower_90} to {upper_90} goals")
print(f"2022 empirical coverage of that interval: {coverage_90:.3f}")

print("\nActual 2022 frequency vs predicted probability:")
print("goals | actual_count | actual_share | predicted_probability")
for row in actual_vs_predicted_rows:
    print(
        f"{row['goals']:>5} | {row['actual_count']:>12} | "
        f"{row['actual_share']:.3f}        | {row['predicted_probability']:.3f}"
    )

try:
    import matplotlib.pyplot as plt
    x = [row["goals"] for row in actual_vs_predicted_rows]
    actual = [row["actual_share"] for row in actual_vs_predicted_rows]
    predicted = [row["predicted_probability"] for row in actual_vs_predicted_rows]

    plt.figure(figsize=(8, 4))
    plt.bar([value - 0.2 for value in x], actual, width=0.4, label="Actual 2022 share")
    plt.bar([value + 0.2 for value in x], predicted, width=0.4, label="Predicted probability")
    plt.title("Actual 2022 Goal Counts vs Posterior Predictive Model")
    plt.xlabel("Total goals in match")
    plt.ylabel("Share / probability")
    plt.legend()
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib unavailable; comparison table printed above.")

## Optional Classification: Over 2.5 Goals

The Bayesian Poisson model is a count model, not a classifier. However, we can create a secondary classification task:

- positive: match has over 2.5 goals, meaning 3 or more goals
- negative: match has 2 or fewer goals

This is where true positive, false positive, true negative, and false negative become meaningful.

In [ ]:
prob_over_2_5 = 1 - sum(
    negative_binomial_predictive_pmf(k, alpha_post, beta_post)
    for k in range(0, 3)
)
predict_over_2_5 = prob_over_2_5 >= 0.5

tp = fp = tn = fn = 0
for actual_goals in test_goals:
    actual_over_2_5 = actual_goals >= 3
    if predict_over_2_5 and actual_over_2_5:
        tp += 1
    elif predict_over_2_5 and not actual_over_2_5:
        fp += 1
    elif not predict_over_2_5 and not actual_over_2_5:
        tn += 1
    else:
        fn += 1

print(f"P(over 2.5 goals) = {prob_over_2_5:.3f}")
print(f"Decision threshold: predict over 2.5 if probability >= 0.5")
print({"TP": tp, "FP": fp, "TN": tn, "FN": fn})

## Optional Comparison: All Earlier World Cups

A useful discussion point is whether old tournaments should be included. Early World Cups had different formats and often higher scoring rates. The code below compares a model trained on all completed tournaments before 2022 with the recent-data model.

In [ ]:
all_prior_years = [
    1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970,
    1974, 1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006,
    2010, 2014, 2018,
]

all_prior_goals = []
for year in all_prior_years:
    all_prior_goals.extend(total_goals_by_match(year))

alpha_all = alpha_prior + sum(all_prior_goals)
beta_all = beta_prior + len(all_prior_goals)
posterior_mean_all = alpha_all / beta_all
comparison_rows = [
    {"model": "2010-2018", "mean": posterior_mean},
    {"model": "1930-2018", "mean": posterior_mean_all},
    {"model": "Actual 2022", "mean": test_average},
]

print(f"Recent-data posterior mean: {posterior_mean:.3f}")
print(f"All-prior-years posterior mean: {posterior_mean_all:.3f}")
print(f"Actual 2022 average: {test_average:.3f}")
print("\nComparison of expected/actual goals per match:")
ascii_bar_chart(comparison_rows, "model", "mean")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 4))
    plt.bar([row["model"] for row in comparison_rows], [row["mean"] for row in comparison_rows])
    plt.title("Recent vs Historical Training Choice")
    plt.ylabel("Goals per match")
    plt.show(block=False)
    plt.close()
except ImportError:
    print("matplotlib unavailable; comparison chart printed above.")

## Interpretation

This notebook demonstrates the full Bayesian workflow:

1. Choose a probability model for count data.
2. Choose a prior for the unknown rate.
3. Update the prior with observed World Cup goals.
4. Use the posterior predictive distribution to reason about future or held-out matches.
5. Discuss limitations honestly.

The result is not a complete football forecasting model. It is a clear statistical model for explaining Bayesian Poisson inference.